In [1]:
!pip -q install lpips torchmetrics pandas pillow

In [2]:
import os, random, shutil, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from google.colab import files
import lpips
from torchmetrics.functional.image import structural_similarity_index_measure

SEED = 42
SPLIT_SEED = 2026
MESSAGE_SEED = 2027
MAX_DELTA = 0.100436
N_IMAGES = 100
MESSAGES_PER_IMAGE = 64
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)


Device: cuda


## 1. Upload zip files


In [4]:
print("1/2: Upload checkpoint ZIP file")
model_upload = files.upload()
model_zip = next((x for x in model_upload if x.lower().endswith('.zip')), None)
assert model_zip, "Model ZIP is necessary."
print("2/2: Upload images")
image_upload = files.upload()
image_zip = next((x for x in image_upload if x.lower().endswith('.zip')), None)
assert image_zip, "Images ZIP is necessary."
MODEL_DIR = Path('/content/final_models'); IMAGE_DIR = Path('/content/final_images')
for directory in (MODEL_DIR, IMAGE_DIR):
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True)
with zipfile.ZipFile(model_zip) as z: z.extractall(MODEL_DIR)
with zipfile.ZipFile(image_zip) as z: z.extractall(IMAGE_DIR)
checkpoints = sorted(p for p in MODEL_DIR.rglob('*.pth') if not p.name.startswith('._'))
print('Checkpoints found:')
for p in checkpoints: print(' -', p.name)


1/2: Upload checkpoint ZIP file


Saving modeller.zip to modeller.zip
2/2: Upload images


Saving train.zip to train.zip
Checkpoints found:
 - selected_by_val_BASELINE_MSE0p1_seed42.pth
 - selected_by_val_CVVDP_w0p1_seed42.pth
 - selected_by_val_FLIP_w0p03_seed42.pth
 - selected_by_val_LPIPS_w10p0_seed42.pth


## 2. Select Checkpoints



In [8]:

CHECKPOINT_NAMES = {
    "Baseline": "selected_by_val_BASELINE_MSE0p1_seed42.pth",
    "LPIPS": "selected_by_val_LPIPS_w10p0_seed42.pth",
    "FLIP": "selected_by_val_FLIP_w0p03_seed42.pth",
    "ColorVideoVDP": "selected_by_val_CVVDP_w0p1_seed42.pth",
}

MODEL_PATHS = {}
for model, name in CHECKPOINT_NAMES.items():
    matches = [p for p in checkpoints if p.name == name]
    if len(matches) != 1:
        raise FileNotFoundError(f'{model}: {name} error. Check the list!')
    MODEL_PATHS[model] = matches[0]
print('Checkpoints ready')


Checkpoints ready


In [9]:

valid = ('.jpg','.jpeg','.png','.webp','.bmp')
image_paths = sorted(str(p) for p in IMAGE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in valid and not p.name.startswith('.'))
assert len(image_paths) >= 900, f'Waiting for images, {len(image_paths)} found.'
rng = random.Random(SPLIT_SEED)
rng.shuffle(image_paths)
metric_paths = image_paths[:N_IMAGES]
generator = torch.Generator().manual_seed(MESSAGE_SEED)
fixed_labels = [torch.randperm(256, generator=generator)[:MESSAGES_PER_IMAGE] for _ in metric_paths]
print('Images:',len(metric_paths),'| Test pair per model:',len(metric_paths)*MESSAGES_PER_IMAGE)


Images: 100 | Test pair per model: 6400


## 3. TAESD and CNNs

In [11]:

TAESD_BASE_URL = "https://raw.githubusercontent.com/madebyollin/taesd/main"

for fname in [
    "taesd.py",
    "taesd_encoder.pth",
    "taesd_decoder.pth"
]:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(
            f"{TAESD_BASE_URL}/{fname}",
            fname
        )

from taesd import TAESD

taesd = TAESD(
    encoder_path="taesd_encoder.pth",
    decoder_path="taesd_decoder.pth"
).to(DEVICE).eval()

for p in taesd.parameters():
    p.requires_grad_(False)

upsample_512 = transforms.Resize(
    (512, 512),
    antialias=True
)

downsample_64 = transforms.Resize(
    (64, 64),
    antialias=True
)

load_transform = transforms.Compose([
    transforms.Resize((64, 64), antialias=True),
    transforms.ToTensor()
])

print("TAESD ready.")


TAESD ready.


In [12]:

class LatentEmbedder(nn.Module):
    def __init__(
        self,
        num_classes=256,
        latent_channels=4,
        max_delta=MAX_DELTA
    ):
        super().__init__()
        self.max_delta = max_delta

        self.message_embedding = nn.Embedding(
            num_classes, 128
        )

        self.message_projection = nn.Sequential(
            nn.Linear(128, 256 * 8 * 8),
            nn.ReLU()
        )

        self.message_upsampler = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU()
        )

        self.delta_network = nn.Sequential(
            nn.Conv2d(latent_channels + 32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, latent_channels, 3, padding=1)
        )

    def forward(self, latent_clean, labels):
        b = latent_clean.size(0)

        msg = self.message_embedding(labels)
        msg = self.message_projection(msg)
        msg = msg.view(b, 256, 8, 8)
        msg = self.message_upsampler(msg)

        raw_delta = self.delta_network(
            torch.cat([latent_clean, msg], dim=1)
        )

        delta = torch.tanh(raw_delta) * self.max_delta
        return latent_clean + delta, delta


class MessageExtractor(nn.Module):
    def __init__(self, num_classes=256):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1),
            nn.GroupNorm(4, 32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, 2, 1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, 2, 1),
            nn.GroupNorm(8, 128),
            nn.ReLU(),

            nn.Conv2d(128, 256, 3, 2, 1),
            nn.GroupNorm(16, 256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.output_layer = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.output_layer(x.flatten(1))


latent_embedder = LatentEmbedder().to(DEVICE)
message_extractor = MessageExtractor().to(DEVICE)

def activate_checkpoint(path):
    ck = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False
    )

    latent_embedder.load_state_dict(
        ck["latent_embedder_state_dict"]
    )
    message_extractor.load_state_dict(
        ck["message_extractor_state_dict"]
    )

    latent_embedder.eval()
    message_extractor.eval()

    return ck

print("Models ready.")


Models ready.


In [13]:

lpips_alex = lpips.LPIPS(net='alex').to(DEVICE).eval()
for p in lpips_alex.parameters(): p.requires_grad_(False)

def load_image(path):
    return load_transform(Image.open(path).convert('RGB'))

def per_image_psnr(x, y):
    mse = ((x - y) ** 2).mean(dim=(1,2,3))
    return 10 * torch.log10(1 / torch.clamp(mse, min=1e-12))


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 233M/233M [00:01<00:00, 244MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth


## 4. Final test

In [14]:
def evaluate_model(model_name, checkpoint_path):
    checkpoint = activate_checkpoint(str(checkpoint_path))
    total = correct = 0
    psnr_sum = ssim_sum = e2e_sum = lpips_sum = 0.0

    with torch.no_grad():
        for i, image_path in enumerate(metric_paths):
            original = load_image(image_path).unsqueeze(0).to(DEVICE)
            z = taesd.encoder(upsample_512(original))
            clean = downsample_64(taesd.decoder(z).clamp(0,1))

            for start in range(0, MESSAGES_PER_IMAGE, BATCH_SIZE):
                labels = fixed_labels[i][start:start+BATCH_SIZE].to(DEVICE)
                batch = labels.size(0)
                z_batch = z.repeat(batch,1,1,1)
                clean_batch = clean.repeat(batch,1,1,1)
                original_batch = original.repeat(batch,1,1,1)
                stego_latent, _ = latent_embedder(z_batch, labels)
                stego = downsample_64(taesd.decoder(stego_latent).clamp(0,1))
                predictions = message_extractor(stego).argmax(1)

                correct += (predictions == labels).sum().item()
                psnr_sum += per_image_psnr(stego, clean_batch).sum().item()
                ssim_sum += structural_similarity_index_measure(stego, clean_batch, data_range=1.0).item() * batch
                e2e_sum += per_image_psnr(stego, original_batch).sum().item()
                lpips_sum += lpips_alex(stego*2-1, clean_batch*2-1).mean().item() * batch
                total += batch
            if (i+1) % 20 == 0:
                print(f'{model_name}: {i+1}/{N_IMAGES} images')

    return {
        'Model': model_name,
        'Epoch': int(checkpoint.get('epoch', -1)),
        'Message Accuracy (%)': 100 * correct/total,
        'PSNR (dB)': psnr_sum/total,
        'SSIM': ssim_sum/total,
        'End-to-end PSNR (dB)': e2e_sum/total,
        'LPIPS': lpips_sum/total,
    }

results = pd.DataFrame([evaluate_model(name,path) for name,path in MODEL_PATHS.items()])
print('\nFINAL TEST — 100 images × 64 messages')
display(results.round({'Message Accuracy (%)':2,'PSNR (dB)':2,'SSIM':4,'End-to-end PSNR (dB)':2,'LPIPS':4}))
results.to_csv('/content/final_test_results.csv', index=False)
print('CSV saved: /content/final_test_results.csv')


Baseline: 20/100 images
Baseline: 40/100 images
Baseline: 60/100 images
Baseline: 80/100 images
Baseline: 100/100 images
LPIPS: 20/100 images
LPIPS: 40/100 images
LPIPS: 60/100 images
LPIPS: 80/100 images
LPIPS: 100/100 images
FLIP: 20/100 images
FLIP: 40/100 images
FLIP: 60/100 images
FLIP: 80/100 images
FLIP: 100/100 images
ColorVideoVDP: 20/100 images
ColorVideoVDP: 40/100 images
ColorVideoVDP: 60/100 images
ColorVideoVDP: 80/100 images
ColorVideoVDP: 100/100 images

FINAL TEST — 100 images × 64 messages


,Model,Epoch,Message Accuracy (%),PSNR (dB),SSIM,End-to-end PSNR (dB),LPIPS
0,Baseline,75,88.62,31.28,0.9206,24.83,0.0337
1,LPIPS,75,89.72,39.27,0.9875,25.68,0.0029
2,FLIP,73,87.25,31.42,0.9319,24.92,0.0209
3,ColorVideoVDP,68,95.80,39.76,0.9891,25.70,0.0026


CSV saved: /content/final_test_results.csv
